# Model Comparison: ResNet50 vs MobileNetV2 vs EfficientNetB0
**Dataset**: CIFAR-10 (subset for Colab free tier)

**Metrics**: Accuracy · Parameters · Training time · Memory

> ⚠️ Training time is **indicative only** — Colab free tier GPU allocation is non-deterministic.

In [ ]:
# Cell 1 — Install & imports
!pip install -q tensorflow psutil

import os, time, gc, psutil
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

print('TF version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

# Limit GPU memory growth to avoid OOM on free tier
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
# Cell 2 — Config
# Colab free tier: keep these small to avoid OOM / timeout
IMG_SIZE    = 96        # 96×96 is enough; 224×224 causes OOM for ResNet50 batch32
BATCH_SIZE  = 32
EPOCHS      = 5         # 5 epochs ≈ signal without full convergence; raise to 10 if stable
N_TRAIN     = 5000      # subset of 50k (10% per class) — full dataset is ~10x slower
N_VAL       = 1000
NUM_CLASSES = 10
SEED        = 42

print(f'Config: {IMG_SIZE}×{IMG_SIZE}px | batch={BATCH_SIZE} | epochs={EPOCHS}')
print(f'Train subset: {N_TRAIN} | Val subset: {N_VAL}')

In [ ]:
# Cell 3 — Load & preprocess CIFAR-10
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.cifar10.load_data()

# Subsample for free tier
np.random.seed(SEED)
train_idx = np.random.choice(len(x_train_full), N_TRAIN, replace=False)
val_idx   = np.random.choice(len(x_test_full),  N_VAL,   replace=False)

x_train = x_train_full[train_idx].astype('float32') / 255.0
y_train = y_train_full[train_idx]
x_val   = x_test_full[val_idx].astype('float32') / 255.0
y_val   = y_test_full[val_idx]

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
print(f'x_train: {x_train.shape}  x_val: {x_val.shape}')

In [ ]:
# Cell 4 — tf.data pipeline with resize + augmentation
def make_dataset(x, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=SEED)
    def preprocess(img, label):
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        if training:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, 0.1)
        return img, tf.squeeze(label)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(x_train, y_train, training=True)
val_ds   = make_dataset(x_val,   y_val,   training=False)
print('Datasets ready')

In [ ]:
# Cell 5 — Model builder (pretrained ImageNet weights, frozen base)
def build_model(base_class, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    """
    Transfer learning setup:
    - Pretrained ImageNet weights
    - Base frozen (only classification head trains)
    - Lightweight head: GlobalAvgPool → Dropout → Dense(10)
    Keeping base frozen means training is fast enough on free tier.
    """
    inputs = layers.Input(shape=input_shape)
    base   = base_class(include_top=False, weights='imagenet', input_tensor=inputs)
    base.trainable = False  # Freeze for speed — unfreeze for final fine-tuning

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print('Builder ready')

In [ ]:
# Cell 6 — Memory tracking helpers
def get_ram_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def get_gpu_mb():
    """Returns GPU memory in MB; returns 0 if no GPU."""
    try:
        info = tf.config.experimental.get_memory_info('GPU:0')
        return info['current'] / 1024**2
    except:
        return 0.0

def count_params(model):
    total     = model.count_params()
    trainable = sum(tf.size(v).numpy() for v in model.trainable_variables)
    return total, trainable

print('Helpers ready')

In [ ]:
# Cell 7 — Training loop (runs all 3 models sequentially)
# Runs sequentially to avoid GPU memory fragmentation on free tier

MODEL_REGISTRY = {
    'ResNet50':      ResNet50,
    'MobileNetV2':   MobileNetV2,
    'EfficientNetB0': EfficientNetB0,
}

results = {}
histories = {}

early_stop = EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True)

for name, base_class in MODEL_REGISTRY.items():
    print(f'\n{'='*50}')
    print(f'Training {name}...')

    # Clear previous model from GPU
    tf.keras.backend.clear_session()
    gc.collect()

    ram_before = get_ram_mb()
    gpu_before = get_gpu_mb()

    model = build_model(base_class)
    total_params, trainable_params = count_params(model)

    t_start = time.time()
    history = model.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=[early_stop],
        verbose=1
    )
    t_end = time.time()

    train_time   = t_end - t_start
    ram_after    = get_ram_mb()
    gpu_after    = get_gpu_mb()
    val_accuracy = max(history.history['val_accuracy'])

    results[name] = {
        'val_accuracy':    round(val_accuracy * 100, 2),
        'total_params':    total_params,
        'trainable_params': trainable_params,
        'train_time_s':    round(train_time, 1),
        'ram_delta_mb':    round(ram_after - ram_before, 1),
        'gpu_delta_mb':    round(gpu_after - gpu_before, 1),
    }
    histories[name] = history.history

    print(f'\n{name} → Acc: {val_accuracy*100:.2f}%  Time: {train_time:.1f}s  '
          f'Params: {total_params/1e6:.1f}M  RAM Δ: {ram_after-ram_before:.1f}MB')

print('\n✅ All models trained')

In [ ]:
# Cell 8 — Results table
import pandas as pd

df = pd.DataFrame(results).T
df['total_params_M']     = (df['total_params'] / 1e6).round(2)
df['trainable_params_M'] = (df['trainable_params'] / 1e6).round(2)

display_cols = [
    'val_accuracy', 'total_params_M', 'trainable_params_M',
    'train_time_s', 'ram_delta_mb', 'gpu_delta_mb'
]
print(df[display_cols].to_string())

In [ ]:
# Cell 9 — Visualize results
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Model Comparison on CIFAR-10 (subset)', fontsize=14, fontweight='bold')

names  = list(results.keys())
colors = ['#4C72B0', '#DD8452', '#55A868']

metrics = [
    ('val_accuracy',    'Validation Accuracy (%)', 'Accuracy (%)'),
    ('total_params_M',  'Total Parameters (M)',    'Params (M)'),
    ('train_time_s',    'Training Time (s)',        'Seconds'),
    ('ram_delta_mb',    'RAM Consumption (MB)',     'MB'),
    ('gpu_delta_mb',    'GPU Memory (MB)',           'MB'),
    ('trainable_params_M', 'Trainable Params (M)',  'Params (M)'),
]

for ax, (col, title, ylabel) in zip(axes.flat, metrics):
    vals = [results[n].get(col, df.loc[n, col]) for n in names]
    bars = ax.bar(names, vals, color=colors, width=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(axis='x', labelsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved as model_comparison.png')

In [ ]:
# Cell 10 — Training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Validation Accuracy per Epoch', fontsize=13)

for ax, (name, hist) in zip(axes, histories.items()):
    ax.plot(hist['accuracy'],     label='Train',  linestyle='--', marker='o')
    ax.plot(hist['val_accuracy'], label='Val',    linestyle='-',  marker='s')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 — Edge deployment scoring
# Normalized score: lower is better for params/time/memory, higher for accuracy
# Weights are opinionated for edge use case

def normalize(vals, invert=False):
    arr = np.array(vals, dtype=float)
    if arr.max() == arr.min():
        return np.ones_like(arr) * 0.5
    norm = (arr - arr.min()) / (arr.max() - arr.min())
    return (1 - norm) if invert else norm

# Weights for edge suitability
W_ACC    = 0.35   # accuracy still matters
W_PARAMS = 0.25   # fewer params = better
W_TIME   = 0.20   # faster = better
W_MEM    = 0.20   # less memory = better

accs   = [results[n]['val_accuracy']   for n in names]
params = [results[n]['total_params']   for n in names]
times  = [results[n]['train_time_s']   for n in names]
mems   = [results[n]['ram_delta_mb']   for n in names]

score_acc    = normalize(accs,   invert=False) * W_ACC
score_params = normalize(params, invert=True)  * W_PARAMS
score_time   = normalize(times,  invert=True)  * W_TIME
score_mem    = normalize(mems,   invert=True)  * W_MEM

edge_scores = score_acc + score_params + score_time + score_mem

print('\n🏆 Edge Deployment Scores (higher = better for edge):')
for name, score in sorted(zip(names, edge_scores), key=lambda x: -x[1]):
    print(f'  {name:20s}: {score:.3f}')

winner = names[np.argmax(edge_scores)]
print(f'\n✅ Recommended for edge: {winner}')

In [ ]:
# Cell 12 — TFLite conversion (edge-ready artifact)
# Re-build winning model and convert to TFLite INT8 (quantized)

tf.keras.backend.clear_session()

winner_class = MODEL_REGISTRY[winner]
final_model  = build_model(winner_class)

# Quick retrain on same data for a real model
final_model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds,
                callbacks=[EarlyStopping(patience=2, restore_best_weights=True)], verbose=0)

# Save SavedModel format first
final_model.save('/tmp/best_model')

# Convert to TFLite (float16 — good balance of size vs accuracy)
converter = tf.lite.TFLiteConverter.from_saved_model('/tmp/best_model')
converter.optimizations      = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

tflite_path = f'/content/{winner}_cifar10.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / 1024**2
print(f'\nTFLite model saved: {tflite_path}')
print(f'TFLite size: {size_mb:.2f} MB')

# Download from Colab
from google.colab import files
files.download(tflite_path)
print('✅ Download triggered')

## Summary & Edge Deployment Notes

| Model | Params | Edge Fit |
|---|---|---|
| ResNet50 | ~25.6M | ❌ Too heavy for edge |
| MobileNetV2 | ~3.4M | ✅ Designed for edge/mobile |
| EfficientNetB0 | ~5.3M | ✅ Best accuracy/size tradeoff |

**Recommendation**: MobileNetV2 for strict memory/latency constraints. EfficientNetB0 when you have ~5MB headroom and want better accuracy.

**Caveats on this experiment**:
- Training time numbers are Colab-GPU-dependent; don't treat them as ground truth.
- 5-epoch frozen-base training is not convergence — accuracy will improve with more epochs or fine-tuning.
- CIFAR-10 images are 32×32 upscaled to 96×96; this harms all models equally but doesn't reflect real-world edge data.
